In [ ]:
from mnist import Net, EuclideanNet, test
from hyptorch.pmath import dist0

import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns

import torch
from torchvision import datasets, transforms
import numpy as np
import os
import json
import dataclasses as dc

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")

np.random.seed(0)
torch.manual_seed(0);

In [ ]:
job_id = "440533"#"440531"

args_paths = []
state_dict_paths = {}

dims = [2, 8, 16, 32]
for dim in dims:
    args_path = f"checkpoints/{job_id}/mnist_args_{dim}d.json"
    args_paths.append(args_path)

checkpoints = os.listdir(f"checkpoints/{job_id}")
for checkpoint in checkpoints:
    if checkpoint.split('.')[-1] == "json":
        continue
    
    dim = int(checkpoint.split('_')[-1].split('.')[0][:-1])
    ep  = int(checkpoint.split('_')[-2])

    state_dict_paths.setdefault(dim, []).append((ep, f"checkpoints/{job_id}/{checkpoint}",))

data_path = "../mnist_data"
if os.getenv("GLOBALSCRATCH") is not None:
    data_path = os.path.join(os.getenv("GLOBALSCRATCH"), "mnist_data")

omni_path = os.path.join(data_path, "omniglot")

In [ ]:
models = {}

for dim, args_path in zip(dims, args_paths):
    with open(args_path, "r") as f:
        args = json.load(f)

    args = dc.make_dataclass("Args", [(k, type(v)) for k, v in args.items()])(**args)

    for ep, checkpoint in state_dict_paths[dim]:
        model = Net(args).to(device)
        state_dict = torch.load(checkpoint, map_location=device)
        model.load_state_dict(state_dict)
        model.eval()
        
        models.setdefault(dim, {})[ep] = model

In [ ]:
kwargs = {"num_workers": 1, "pin_memory": True} if device.type == "cuda" else {}
train_MNIST_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        data_path,
        train=True,
        download=True,
        transform=transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        ),
    ),
    batch_size=args.batch_size,
    shuffle=True,
    **kwargs
)

test_MNIST_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        data_path,
        train=False,
        transform=transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        ),
    ),
    batch_size=args.test_batch_size,
    shuffle=True,
    **kwargs
)

In [ ]:
test_Omniglot_loader = torch.utils.data.DataLoader(
    datasets.Omniglot(
        omni_path,
        background=False,
        download=True,
        transform=transforms.Compose(
            [
                transforms.Resize((28, 28)), 
                transforms.ToTensor(),
                transforms.Lambda(lambda image: 1.0 - image),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        ),
    ),
    batch_size=args.test_batch_size,
    shuffle=True,
    **kwargs
)

In [ ]:
for dim in dims:
    for ep, model in models[dim].items():
        loss, _ = test(None, model, device, test_MNIST_loader)
        models[dim][ep] = (model, loss)

In [ ]:
def collect_dist0(model, loader, device):
    model.eval()
    distances = []
    targets = []

    with torch.no_grad():
        for data, target in loader:
            data = data.to(device)
            embedding = model.embed(data)
            dist = dist0(embedding, c=model.tp.c).view(-1)
            distances.extend(dist.cpu().numpy())
            targets.extend(target.cpu().numpy())
            
    return np.array(distances), np.array(targets)

In [ ]:
dim = 16
MNIST_dists_by_label = []
losses = []

for ep, (model, loss) in list(models[dim].items())[-20:]:
    dists, targets = collect_dist0(model, test_MNIST_loader, device)

    MNIST_dists_by_label.append([])
    for label in range(10):
        MNIST_dists_by_label[-1].append(dists[targets==label])

    losses.append(loss)

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

checkpoint_epochs = [epoch for epoch, _ in list(models[dim].items())[-20:]]
distribution_frames = []
checkpoint_labels = []

for epoch, loss, distances_by_label in zip(
    checkpoint_epochs, losses, MNIST_dists_by_label
):
    checkpoint_label = f"Epoch {epoch} (loss={loss:.5f})"
    checkpoint_labels.append(checkpoint_label)

    for label, distance_values in enumerate(distances_by_label):
        distribution_frames.append(
            pd.DataFrame(
                {
                    "distance": distance_values,
                    "label": str(label),
                    "checkpoint": checkpoint_label,
                }
            )
        )

ridge_data = pd.concat(distribution_frames, ignore_index=True)
sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

label_palette = dict(zip(map(str, range(10)), sns.color_palette("tab10", 10)))
digit_checkboxes = [
    widgets.Checkbox(value=(label == 0), description=str(label), indent=False)
    for label in range(10)
]
plot_output = widgets.Output()


def update_digit_plot(change=None):
    selected_labels = [
        checkbox.description for checkbox in digit_checkboxes if checkbox.value
    ]

    with plot_output:
        clear_output(wait=True)
        if not selected_labels:
            print("Select at least one digit to display its distributions.")
            return

        selected_data = ridge_data[ridge_data["label"].isin(selected_labels)]
        g = sns.FacetGrid(
            selected_data,
            row="checkpoint",
            hue="label",
            row_order=checkpoint_labels,
            hue_order=selected_labels,
            palette=label_palette,
            aspect=14,
            height=0.7,
            sharex=True,
            sharey=False,
        )
        g.map(
            sns.kdeplot,
            "distance",
            fill=True,
            alpha=0.45,
            linewidth=1.1,
            bw_adjust=0.8,
            clip_on=False,
            common_norm=False,
        )
        g.refline(y=0, linewidth=1, linestyle="-", color="0.35", clip_on=False)

        g.set_titles("")
        for ax, checkpoint_label in zip(g.axes.flat, checkpoint_labels):
            ax.text(
                -0.02,
                0.08,
                checkpoint_label,
                transform=ax.transAxes,
                ha="right",
                va="center",
                fontsize=12,
                clip_on=False,
            )

        g.set(yticks=[], ylabel="")
        g.set_xlabels("Distance to origin")
        g.add_legend(title="Digit", fontsize=12)
        g.despine(bottom=True, left=True)
        g.figure.suptitle(
            f"MNIST distance-to-origin distributions by digit across checkpoints (dim={dim})",
            y=0.99,
        )
        g.figure.subplots_adjust(top=0.94, left=0.25, hspace=-0.4)
        g.figure.savefig(f"exp_images/evolution_of_d2o_by_label_d{dim}.svg")
        display(g.figure)
        plt.close(g.figure)


for checkbox in digit_checkboxes:
    checkbox.observe(update_digit_plot, names="value")

display(widgets.VBox([widgets.HBox(digit_checkboxes), plot_output]))
update_digit_plot()

In [ ]:
dim = 16
MNIST_dists = []
omni_dists = []
losses = []

for ep, (model, loss) in list(models[dim].items())[-20:]:
    MNIST_dists.append(collect_dist0(model, test_MNIST_loader, device))
    omni_dists.append(collect_dist0(model, test_Omniglot_loader, device))
    losses.append(loss)

In [ ]:
import pandas as pd
from scipy.stats import gaussian_kde
from matplotlib.patches import ConnectionPatch

checkpoint_epochs = [epoch for epoch, _ in list(models[dim].items())[-20:]]
distribution_frames = []
checkpoint_labels = []

for epoch, loss, mnist_dist, omni_dist in zip(
    checkpoint_epochs, losses, MNIST_dists, omni_dists
):
    checkpoint_label = f"Epoch {epoch} (loss={loss:.5f})"
    checkpoint_labels.append(checkpoint_label)

    for dataset_name, distance_values in (
        ("MNIST", mnist_dist),
        ("Omniglot", omni_dist),
    ):
        distribution_frames.append(
            pd.DataFrame(
                {
                    "distance": distance_values,
                    "dataset": dataset_name,
                    "checkpoint": checkpoint_label,
                }
            )
        )

ridge_data = pd.concat(distribution_frames, ignore_index=True)
sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

palette = {"MNIST": "#d55e00", "Omniglot": "#0072b2"}
g = sns.FacetGrid(
    ridge_data,
    row="checkpoint",
    hue="dataset",
    row_order=checkpoint_labels,
    hue_order=["MNIST", "Omniglot"],
    palette=palette,
    aspect=14,
    height=0.7,
    sharex=True,
    sharey=False,
)
g.map(
    sns.kdeplot,
    "distance",
    fill=True,
    alpha=0.65,
    linewidth=1.25,
    bw_adjust=0.8,
    clip_on=False,
    common_norm=False,
)
g.refline(y=0, linewidth=1, linestyle="-", color="0.35", clip_on=False)

for dataset_name, distances_by_checkpoint in (
    ("MNIST", MNIST_dists),
    ("Omniglot", omni_dists),
):
    mean_points = []
    for ax, distances in zip(g.axes.flat, distances_by_checkpoint):
        mean = float(np.mean(distances))
        kde = gaussian_kde(
            distances,
            bw_method=lambda estimator: estimator.scotts_factor() * 0.8,
        )
        density_at_mean = float(kde(mean)[0])
        ax.scatter(
            mean,
            density_at_mean,
            color=palette[dataset_name],
            edgecolor="white",
            linewidth=0.7,
            s=38,
            zorder=10,
        )
        mean_points.append((ax, mean, density_at_mean))

    for (ax_a, mean_a, density_a), (ax_b, mean_b, density_b) in zip(
        mean_points, mean_points[1:]
    ):
        connector = ConnectionPatch(
            xyA=(mean_a, density_a),
            xyB=(mean_b, density_b),
            coordsA="data",
            coordsB="data",
            axesA=ax_a,
            axesB=ax_b,
            color=palette[dataset_name],
            linewidth=1.6,
            zorder=9,
        )
        connector.set_clip_on(False)
        g.figure.add_artist(connector)

g.set_titles("")
for ax, checkpoint_label in zip(g.axes.flat, checkpoint_labels):
    ax.text(
        -0.02,
        0.08,
        checkpoint_label,
        transform=ax.transAxes,
        ha="right",
        va="center",
        fontsize=12,
        clip_on=False,
    )

g.set(yticks=[], ylabel="")
g.set_xlabels("Distance to origin")
g.add_legend(title="Dataset", fontsize=12)
g.despine(bottom=True, left=True)
g.figure.suptitle(f"Distance-to-origin distributions across checkpoints (dim={dim})", y=0.99)
g.figure.subplots_adjust(top=0.94, left=0.25, hspace=-0.4)

plt.savefig(f"exp_images/evolution_of_d2o_d{dim}.svg")

In [ ]:
data, target = next(iter(test_MNIST_loader))
data, target = data.to(device), target.to(device)

In [ ]:
dim, ep = 16, 22
m, loss = models[dim][ep]
output = m(data)

with torch.no_grad():
    num_samples = 10000
    sample_data = data.detach().cpu().numpy()[:num_samples]
    sample_target = target.detach().cpu().numpy()[:num_samples]
    sample_embeddings = m.embed(data)[:num_samples]
    sample_activations = m(data)[:num_samples].detach().cpu().numpy()

In [ ]:
os.makedirs("mnist_samples", exist_ok=True)
np.save(f"mnist_samples/mnist_embeddings_{dim}d_{ep}e.npy", sample_embeddings.detach().cpu().numpy())
np.save(f"mnist_samples/mnist_target_{dim}d_{ep}e.npy", sample_target)
np.save(f"mnist_samples/mnist_data_{dim}d_{ep}e.npy", sample_data)
np.save(f"mnist_samples/mnist_activations_{dim}d_{ep}e.npy", sample_activations)